## Correct respondent locations

Assigns every respondent to one of the **sampled** clusters (`DIR_SETTLEMENTS_SAMPLED`, the 58 clusters drawn), which gives them the cluster's name, department, population and PPS proportion. The full eligible frame isn't needed here (02b uses it for stratum totals).

1. **Schedule → sample**: each selected settlement in the field schedule (`DIR_FIELD_SCHEDULE`) is placed in the sampled cluster its `GPS coordinate` falls in (or the nearest within `SNAP_DISTANCE_M`); without a coordinate, by name.
2. **Respondents** (see `assign_sampled_clusters` in `src/locations.py`). The **spatial join** takes precedence over the manual review's name:
   - **override**: `data/meta/manual_location_overrides.csv` (`KEY, location` = sampled cluster name)
   - **gps**: the GPS point lies inside a sampled cluster polygon
   - **name**: otherwise the manual-review location (applied in 01) is a schedule settlement (→ its sampled cluster), or a sampled cluster's name / one of its villages
   - **gps_snap**: otherwise the nearest sampled cluster, if within `SNAP_DISTANCE_M`
   - **unresolved**: exported to `4_validation_checks/02a_unresolved_locations.csv` and kept in the output without a cluster (02b leaves them out of the weighting)

   (`spatial_first=False` in `assign_sampled_clusters` restores name-first, with GPS winning only beyond `DISAGREE_DISTANCE_M`.)

`location` and `province` stay as the manual review recorded them. Output (`DIR_LOCATED_DATA`) adds `sample_cluster` (the join key for 02b), `sample_province`, `sample_population`, `sample_proportion` and `location_source`. The checks list each manual-review settlement/department against its sampled cluster/department, with `selected_name` (spatial-join name, else the manual-review name), both sides' settlement coordinates and the distance between them, plus where the GPS and the manual-review name disagree.

In [ ]:
import pandas as pd
import geopandas as gpd
from src.config import *
from src.funcs import load_surveyed_clusters, read_table
from src.locations import assign_sampled_clusters, link_schedule

In [ ]:
SNAP_DISTANCE_M = 3000       # a point this close to a sampled cluster (but outside it) is placed in it
DISAGREE_DISTANCE_M = 4000   # GPS inside another cluster AND this far from the named one = clear disagreement

Load the sampled clusters, the field schedule, mapped survey data and any researcher overrides

In [ ]:
sample = load_surveyed_clusters(DIR_SETTLEMENTS_SAMPLED, rename=SETTLEMENTS_SAMPLED_RENAME)
schedule = read_table(DIR_FIELD_SCHEDULE)
data = pd.read_csv(DIR_MAPPED_DATA)

overrides = None
if DIR_MANUAL_LOCATION_OVERRIDES.exists():
    overrides = pd.read_csv(DIR_MANUAL_LOCATION_OVERRIDES, dtype=str).set_index('KEY')['location']

print(f"{len(sample)} sampled clusters | {schedule['Settlement'].nunique()} schedule settlements | {len(data)} respondents | "
      f"{0 if overrides is None else data['KEY'].isin(overrides.index).sum()} with a manual override")

#### Schedule settlements → sampled clusters

In [ ]:
schedule_links = link_schedule(schedule, sample, snap_m=SNAP_DISTANCE_M, utm_crs=UTM_CRS)
schedule_links.to_csv(DIR_VALIDATE / "02a_schedule_to_sample.csv", index=False, encoding='utf-8-sig')
print(f"{schedule_links['sample_cluster'].notna().sum()} of {len(schedule_links)} schedule settlements linked to a sampled cluster: "
      f"{schedule_links['how'].value_counts(dropna=False).to_dict()}")
if schedule_links['conflicting_days'].any():
    print(f"⚠️  Settlements whose GPS points to different clusters on different days (first used): "
          f"{schedule_links.loc[schedule_links['conflicting_days'], 'settlement'].tolist()}")
unvisited = sorted(set(sample['Name']) - set(schedule_links['sample_cluster'].dropna()))
print(f"Sampled clusters no schedule settlement falls in: {unvisited}")
print(schedule_links.assign(dist_m=schedule_links['dist_m'].round(0)).to_string(index=False))

#### Respondents → sampled clusters

In [ ]:
res = assign_sampled_clusters(
    data, sample, schedule_links, overrides=overrides,
    snap_m=SNAP_DISTANCE_M, disagree_m=DISAGREE_DISTANCE_M, utm_crs=UTM_CRS,
)
meta = sample.drop(columns='geometry').set_index('Name')
res['sample_province'] = res['sample_cluster'].map(meta['province'])
unresolved = res['location_source'] == 'unresolved'
res['location_source'].value_counts()

#### Checks (place names and counts only)

In [ ]:
# Manual review (settlement + department) -> sampled cluster (settlement + department), with each side's
# settlement coordinates: the field schedule's GPS coordinate for the manual-review settlement, and the
# sampled cluster's own point (settlement-level points only -- never respondents' GPS).
# selected_name: the spatial-join name (sampled cluster the GPS point lies in), else the manual-review name.
from src.locations import normalize_place, parse_point

review_points = {}
for name, value in zip(schedule['Settlement'], schedule['GPS coordinate']):
    p = parse_point(value)
    if p and isinstance(normalize_place(name), str):
        review_points.setdefault(normalize_place(name), p)   # first usable coordinate per settlement

if {'latitude', 'longitude'} <= set(meta.columns):
    sample_points = {n: (la, lo) for n, la, lo in zip(meta.index, meta['latitude'], meta['longitude'])}
else:   # fall back to a point inside each sampled polygon
    centre = sample.set_index('Name').geometry.representative_point()
    sample_points = {n: (pt.y, pt.x) for n, pt in centre.items()}

mapping = (data[['location', 'province']]
           .join(res[['sample_cluster', 'sample_province', 'location_source']])
           .assign(selected_name=res['cluster_by_gps'].fillna(data['location']))
           .groupby(['location', 'province', 'sample_cluster', 'sample_province', 'location_source', 'selected_name'],
                    dropna=False)
           .size().rename('n').reset_index())
review = mapping['location'].map(lambda n: review_points.get(normalize_place(n)) if isinstance(n, str) else None)
sampled = mapping['sample_cluster'].map(sample_points)
mapping['review_lat'], mapping['review_lon'] = review.map(lambda p: p[0] if p else None), review.map(lambda p: p[1] if p else None)
mapping['sample_lat'], mapping['sample_lon'] = sampled.map(lambda p: p[0] if isinstance(p, tuple) else None), sampled.map(lambda p: p[1] if isinstance(p, tuple) else None)

# Distance between the two points (km), to spot settlements far from their sampled cluster
has_both = mapping[['review_lat', 'sample_lat']].notna().all(axis=1)
mapping['review_to_sample_km'] = None
if has_both.any():
    a = gpd.GeoSeries(gpd.points_from_xy(mapping.loc[has_both, 'review_lon'], mapping.loc[has_both, 'review_lat']), crs='epsg:4326').to_crs(UTM_CRS)
    b = gpd.GeoSeries(gpd.points_from_xy(mapping.loc[has_both, 'sample_lon'], mapping.loc[has_both, 'sample_lat']), crs='epsg:4326').to_crs(UTM_CRS)
    mapping.loc[has_both, 'review_to_sample_km'] = (a.distance(b, align=False) / 1000).round(1).to_numpy()

mapping.to_csv(DIR_VALIDATE / "02a_manual_review_vs_sample.csv", index=False, encoding='utf-8-sig')
pd.set_option('display.max_rows', 200)
print(mapping.to_string(index=False))

dept_diff = mapping.dropna(subset=['sample_province'])
dept_diff = dept_diff[dept_diff['province'].astype(str).str.strip() != dept_diff['sample_province']]
print(f"\n{dept_diff['n'].sum()} respondents whose manual-review department differs from their sampled cluster's department")

In [ ]:
# Spatial join vs manual review: GPS inside a different sampled cluster than the manual-review name points to
# (the spatial join is used)
dis = data[['location']].join(res)[res['name_disagrees']]
print(f"{len(dis)} respondents whose GPS lies in a different sampled cluster than their manual-review name (GPS used):")
print(dis.groupby(['location', 'cluster_by_name', 'cluster_by_gps'])['named_dist_m']
         .agg(n='size', median_km=lambda s: round(s.median() / 1000, 1)).reset_index().to_string(index=False))

In [ ]:
# Unresolved: the name links to no sampled cluster and the GPS is more than SNAP_DISTANCE_M from all of them
u = data[['location', 'province']].join(res)[unresolved]
print(f"{len(u)} unresolved respondents. Distance to the nearest sampled cluster:")
print(pd.cut(u['gps_dist_m'] / 1000, [0, 2, 5, 10, 50, float('inf')],
             labels=['1-2 km', '2-5 km', '5-10 km', '10-50 km', '50+ km']).value_counts().sort_index().to_string())
print(u.groupby(['location', 'province'], dropna=False)['gps_dist_m']
       .agg(n='size', median_nearest_km=lambda s: round(s.median() / 1000, 1)).reset_index().to_string(index=False))

# Rows for manual review (add resolvable ones to manual_location_overrides.csv)
data.loc[unresolved].join(res.loc[unresolved, ['gps_dist_m']]).to_csv(
    DIR_VALIDATE / "02a_unresolved_locations.csv", index=False, encoding='utf-8-sig')

#### Apply and export

In [ ]:
# Final cluster per respondent (`cluster_name`): the researcher-reviewed `selected_name` from
# 02a_manual_review_vs_sample_reviewed.csv, matched on that file's other columns. Respondents with no
# reviewed row keep the rule the file was built with (spatial-join name, else the manual-review name).
from src.locations import normalize_place

keys = ['location', 'province', 'sample_cluster', 'sample_province', 'location_source']
resp = data[['location', 'province']].join(res[['sample_cluster', 'sample_province', 'location_source']])
resp['selected_name'] = res['cluster_by_gps'].fillna(data['location'])

review_path = next((p for p in [DIR_LOCATION_REVIEW, DIR_VALIDATE / DIR_LOCATION_REVIEW.name] if p.exists()), None)
if review_path is None:
    print(f"⚠️  {DIR_LOCATION_REVIEW.name} not found -- using the unreviewed selected_name")
    resp['cluster_name'] = resp['selected_name']
else:
    as_key = lambda df: df[keys].fillna('').astype(str).apply(lambda c: c.str.strip()).agg('|'.join, axis=1)
    reviewed = read_table(review_path)
    reviewed['_k'] = as_key(reviewed)
    if reviewed['_k'].duplicated().any():
        print(f"⚠️  {reviewed['_k'].duplicated().sum()} reviewed rows repeat a location/cluster combination -- first used")
    reviewed_name = as_key(resp).map(reviewed.drop_duplicates('_k').set_index('_k')['selected_name'])
    changed = reviewed_name.notna() & (reviewed_name != resp['selected_name'])
    print(f"Reviewed names ({review_path.parent.name}/{review_path.name}) applied to {reviewed_name.notna().sum()} of "
          f"{len(resp)} respondents; {changed.sum()} changed by the review; "
          f"{reviewed_name.isna().sum()} with no reviewed row (unreviewed selected_name kept)")
    resp['cluster_name'] = reviewed_name.fillna(resp['selected_name'])

# One row per final cluster, with a settlement-level point for 02b to link it to the eligible frame:
# the sampled polygon's centre for a sampled cluster, else the schedule's GPS coordinate for that settlement
centre = sample.set_index('Name').geometry.representative_point()
named = resp.dropna(subset=['cluster_name'])
rows = []
for k, g in named.groupby(named['cluster_name'].map(normalize_place)):
    name = g['cluster_name'].iloc[0]
    if name in centre.index:
        province, point = meta.at[name, 'province'], f"{centre[name].y}, {centre[name].x}"
    else:
        p = review_points.get(k)
        province = g['province'].mode().iat[0] if g['province'].notna().any() else None
        point = f"{p[0]}, {p[1]}" if p else None
    rows.append(dict(Name=name, province=province, point=point, respondents=len(g),
                     sampled_cluster=name in centre.index))
clusters = pd.DataFrame(rows)
clusters.to_csv(DIR_SETTLEMENTS_CHOSEN, index=False, encoding='utf-8-sig')
print(f"{len(clusters)} final clusters ({clusters['sampled_cluster'].sum()} sampled, "
      f"{(~clusters['sampled_cluster']).sum()} other settlements; {clusters['point'].isna().sum()} without a point, "
      f"linked by name in 02b) -> {DIR_SETTLEMENTS_CHOSEN}")
print(clusters.drop(columns='point').to_string(index=False))

located = data.copy()
located['cluster_name'] = resp['cluster_name']
located['sample_cluster'] = res['sample_cluster']
located['sample_province'] = res['sample_province']
located['sample_population'] = res['sample_cluster'].map(meta['population'])
located['sample_proportion'] = res['sample_cluster'].map(meta['proportion'])
located['location_source'] = res['location_source']

located.to_csv(DIR_LOCATED_DATA, index=False)
print(f"Wrote {len(located)} respondents to {DIR_LOCATED_DATA} ({unresolved.sum()} unresolved by GPS/name rules, "
      f"{located['cluster_name'].isna().sum()} without any cluster)")